In [10]:
import requests
from bs4 import BeautifulSoup as bs

In [11]:
base_url = "https://health-products.canada.ca/dpd-bdpp/dispatch-repartition"

resp = requests.get(base_url)
resp.raise_for_status()

content = resp.content
soup = bs(content, "html.parser")
print(soup.prettify())

<!DOCTYPE HTML>
<!--[if gt IE 8]><!-->
<!--[if lt IE 9]><html class="no-js lt-ie9" dir="ltr"><![endif]-->
<html class="no-js" dir="ltr" lang="en" xmlns="http://www.w3.org/1999/xhtml">
 <!--<![endif]-->
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <!-- Web Experience Toolkit (WET) / Boîte à outils de l'expérience Web (BOEW)
         wet-boew.github.io/wet-boew/License-en.html / wet-boew.github.io/wet-boew/Licence-fr.html -->
  <title>
   Search criteria - Drug Product Database online query
  </title>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <!-- Additional analytics, if required. -->
  <!-- Add analytics here -->
  <!-- Load closure template scripts -->
  <script src="https://www.canada.ca/etc/designs/canada/cdts/gcweb/v5_0_0/cdts/compiled/soyutils.js" type="text/javascript">
  </script>
  <script src="https://www.canada.ca/etc/designs/canada/cdts/gcweb/v5_0_0/cdts/compiled/wet-en.js" type="text/javascript">

In [13]:
# Print out all links on the page
for link in soup.find_all("a"):
    href = link.get("href")
    text = link.get_text(strip=True)
    print(text, "→", href)


Skip to main content → #wb-cont
Skip to "About government" → #wb-info
/Gouvernement du Canada → https://www.canada.ca/en.html
Jobs and the workplace → https://www.canada.ca/en/services/jobs.html
Immigration and citizenship → https://www.canada.ca/en/services/immigration-citizenship.html
Travel and tourism → https://travel.gc.ca/
Business and industry → https://www.canada.ca/en/services/business.html
Benefits → https://www.canada.ca/en/services/benefits.html
Health → https://www.canada.ca/en/services/health.html
Taxes → https://www.canada.ca/en/services/taxes.html
Environment and natural resources → https://www.canada.ca/en/services/environment.html
National security and defence → https://www.canada.ca/en/services/defence.html
Culture, history and sport → https://www.canada.ca/en/services/culture.html
Policing, justice and emergencies → https://www.canada.ca/en/services/policing.html
Transport and infrastructure → https://www.canada.ca/en/services/transport.html
Canada and the world → h

# with Selenium

In [1]:
import json
import time
import pandas as pd
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import TimeoutException
import subprocess

# User agents to rotate through
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15"
]

In [2]:
def get_href_safe(parent_element):
    attempts = 3
    while attempts > 0:
        try:
            link_element = parent_element.find_element(By.TAG_NAME, "a")
            href = link_element.get_attribute("href")
            return href
        except StaleElementReferenceException:
            attempts -= 1
            time.sleep(0.5)  # small delay before retry
    return None 

def save_intermediate_results(results, filename="scraping_results.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

def random_delay(min_delay=1, max_delay=5):
    """Add random delay to mimic human behavior"""
    delay = random.uniform(min_delay, max_delay)
    time.sleep(delay)

def setup_driver_with_stealth():
    """Setup Chrome driver with stealth options"""
    options = Options()
    
    # Random user agent
    user_agent = random.choice(USER_AGENTS)
    options.add_argument(f"user-agent={user_agent}")
    
    # Stealth options
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-extensions")
    options.add_argument("--window-size=1920,1080")

    
    driver = webdriver.Chrome(options=options)
    
    # Execute script to remove webdriver property
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

# ---- CONFIG ----
BASE_URL = "https://health-products.canada.ca/dpd-bdpp/search"
OUTPUT_PATH = "./../../data/HealthCanada/"
OUTPUT_CSV =  OUTPUT_PATH + "scraping_results.csv"
WAIT_TIME = 300  # Increased wait time
SLEEP_BETWEEN_PAGES = random.uniform(2, 5)  # Random delay between pages
COUNTER = 0
PDF_COUNTER = 0
UNIQUE_COUNTER = 0
PDF_DICT = {}

# ----------------

# Setup driver with stealth options
driver = setup_driver_with_stealth()

try:
    driver.get(BASE_URL)
    random_delay(1, 3)  # Random delay after page load

    # Wait for page to load and dropdowns present
    WebDriverWait(driver, WAIT_TIME).until(
        EC.presence_of_element_located((By.ID, "drugClass"))
    )

    random_delay(0.5, 1)  # Random delay before interacting

    # Set filters BEFORE clicking Search
    # Handle multi-select Class dropdown: select Human + Radiopharmaceutical
    class_dropdown = Select(driver.find_element(By.ID, "drugClass"))
    class_dropdown.deselect_all()
    random_delay(0.4, 0.8)
    class_dropdown.select_by_visible_text("Human")
    random_delay(0.4, 0.8)
    class_dropdown.select_by_visible_text("Radiopharmaceutical")

    random_delay(1, 2)  # Random delay before search

    try:
        # Wait for the input with value 'Search' to be clickable
        search_button = WebDriverWait(driver, WAIT_TIME).until(
            EC.element_to_be_clickable((By.XPATH, "//input[@value='Search' and @type='submit']"))
        )
        search_button.click()
        print("Clicked the correct search input button.")
        random_delay(0.5, 2.5)  # Random delay after search

    except Exception as e:
        print(f"Could not find or click the correct search button: {e}")
        driver.quit()
        exit()

    results = []
    main_window = driver.current_window_handle  # save main window handle

    while True:
        # Wait until table rows appear
        WebDriverWait(driver, WAIT_TIME).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
        )

        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

        for i in range(len(rows)):
            # Random delay between processing each row
            random_delay(0.1, 0.5)
            
            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            row = rows[i]
            cols = row.find_elements(By.TAG_NAME, "td")
            if not cols:
                continue

            status = cols[0].text.strip()
            din_col = cols[1]
            din = din_col.text.strip()

            detail_link = get_href_safe(din_col)
            if not detail_link:
                print("Failed to get detail link due to stale element")
                continue

            # Open new tab etc.
            driver.execute_script("window.open(arguments[0]);", detail_link)
            driver.switch_to.window(driver.window_handles[-1])
            
            random_delay(0.4, 0.8)  # Random delay after opening new tab

            try:
                # Wait up to 10 seconds for the PDF link to appear
                pdf_link_element = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.XPATH, "//a[contains(@href, '.PDF')]"))
                )
                pdf_url = pdf_link_element.get_attribute("href")
                pdf_filename = pdf_url.split("/")[-1]
                pdf_filename = pdf_filename.split(".")[0]
                print(f"Found PDF URL: {pdf_url} with file name: {pdf_filename}")
            except TimeoutException:
                pdf_url = None
                print("No PDF link found on this page.")

            finally:
                # Close detail tab & return to results tab
                driver.close()
                driver.switch_to.window(main_window)
                random_delay(0.1, 0.5)  # Random delay after closing tab

            COUNTER += 1

            if pdf_url:
                # get pdf with curl and save to <DIN>.pdf
                subprocess.run(["curl", "-o", f"./../../data/HealthCanada/downloads/{pdf_filename}.pdf", pdf_url])
                PDF_COUNTER += 1
                unique_url = pdf_url not in [result["PDF_URL"] for result in results]
                if unique_url:
                    UNIQUE_COUNTER += 1
                pdf_filename = pdf_url.split("/")[-1]
                results.append({"DIN": din, "PDF_URL": pdf_url, "PDF_NAME": pdf_filename, "Status": status, "idx": COUNTER, "PDF_Counter": PDF_COUNTER, "PDF_Counter_Unique": UNIQUE_COUNTER})
                if pdf_url not in PDF_DICT.keys():
                    PDF_DICT[pdf_url] = [din]
                else:
                    PDF_DICT[pdf_url].append(din)
            else:
                results.append({"DIN": din, "PDF_URL": pdf_url, "Status": status, "idx": COUNTER, "PDF_Counter": None, "PDF_Counter_Unique": None})

            # Save intermediate results after each item
            save_intermediate_results(results)

            # save din dict as json
            with open(OUTPUT_PATH + "pdf_dict.json", "w", encoding="utf-8") as f:
                json.dump(PDF_DICT, f, indent=4)

        try:
            next_button = driver.find_element(By.LINK_TEXT, "Next")
            next_button.click()

            WebDriverWait(driver, WAIT_TIME).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "table"))
            )
            WebDriverWait(driver, WAIT_TIME).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
            )
            
            # Random delay between pages
            random_delay(0.3, 0.8)
            
        except TimeoutException:
            with open("timeout_page.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            break
        except Exception:
            break

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    # Close browser
    driver.quit()

    # Save final results CSV
    if 'results' in locals():
        df = pd.DataFrame(results)
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Saved {len(df)} DIN–PDF pairs to {OUTPUT_CSV}")

Clicked the correct search input button.
No PDF link found on this page.
No PDF link found on this page.
No PDF link found on this page.
Found PDF URL: https://pdf.hres.ca/dpd_pm/00034399.PDF with file name: 00034399


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  486k  100  486k    0     0   244k      0  0:00:01  0:00:01 --:--:--  244k


Found PDF URL: https://pdf.hres.ca/dpd_pm/00034399.PDF with file name: 00034399


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  486k  100  486k    0     0   333k      0  0:00:01  0:00:01 --:--:--  332k


Found PDF URL: https://pdf.hres.ca/dpd_pm/00061808.PDF with file name: 00061808


...

Found PDF URL: https://pdf.hres.ca/dpd_pm/00056462.PDF with file name: 00056462


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  858k  100  858k    0     0   340k      0  0:00:02  0:00:02 --:--:--  340k


Found PDF URL: https://pdf.hres.ca/dpd_pm/00032463.PDF with file name: 00032463


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 57148  100 57148    0     0  95805      0 --:--:-- --:--:-- --:--:-- 95885


An error occurred: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=139.0.7258.67); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x000000010336b26c cxxbridge1$str$ptr + 2741972
1   chromedriver                        0x00000001033631dc cxxbridge1$str$ptr + 2709060
2   chromedriver                        0x0000000102ead4fc cxxbridge1$string$len + 90520
3   chromedriver                        0x0000000102eb31b4 cxxbridge1$string$len + 114256
4   chromedriver                        0x0000000102eb54f8 cxxbridge1$string$len + 123284
5   chromedriver                        0x0000000102eb55a0 cxxbridge1$string$len + 123452
6   chromedriver                        0x0000000102eef9d0 cxxbridge1$string$len + 362092
7   chromedriver                        0x0000000102eea2e0 cxxbridge1$strin